# 과제 10 · 02 · 매뉴얼을 절 단위로 자르고 인용 번호 붙이기

## 현업 시나리오
보고서는 "MC01-MM 4.2.3 점검 절차에 따라"처럼 매뉴얼의 **절 번호**로 근거를 대야 합니다(SOP-EA-01 5.3). 그런데 매뉴얼을 글자 수로만 자르면 한 조각에 여러 절이 섞이고, 어느 절을 인용해야 할지 알 수 없습니다.

## 학습 목표
- 글자 수로 자르면 절이 섞이는 것을 숫자로 본다.
- `MarkdownHeaderTextSplitter` 로 절 단위로 자르고, 머리말에서 인용 번호를 만든다.
- 청크 앞에 머리말 경로를 붙이면 BM25 가 절을 찾는 것을 확인한다.
- PDF 에서 읽은 글자로는 같은 일을 할 수 없는 것을 확인한다.

## 직접 조립
완성된 App을 가져오지 않습니다. 아래에서 작은 fixture와 핵심 함수를 직접 만듭니다.

### 1단계 · 글자 수로만 자르기

In [ ]:
# 연습용 매뉴얼입니다. 실제 매뉴얼(MC01-MM)의 4장 일부를 줄인 것입니다.
# RecursiveCharacterTextSplitter 는 글자 수를 지키며 문단·줄·공백에서 자릅니다.
# 관찰 포인트: 절 번호가 정확히 하나인 조각만 "이 절"이라고 인용할 수 있습니다. 둘이거나 없으면 인용할 수 없습니다.
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

PRACTICE_MANUAL = """# MC-01 정비 매뉴얼
## 4. 고장 유형별 진단
### 4.1 공구 마모 고장 (TWF)
#### 4.1.2 판정 기준
공구 마모 시간이 200분 이상이면 위험 구간이다. 240분을 넘기면 사용 한계다.
#### 4.1.3 점검 절차
1. 공구 홀더에서 공구를 분리한다. 2. 날 끝 마모 폭을 측정한다. 3. 마모 폭이 0.3mm 를 넘으면 공구(PN-TL-1008)를 교체한다.
### 4.2 열 방산 고장 (HDF)
#### 4.2.2 판정 기준
공정 온도와 공기 온도의 차이가 8.6 K 미만이고 회전수가 1,380 rpm 미만이면 해당한다.
#### 4.2.3 점검 절차
1. 흡기 필터(PN-FL-3120)의 압력차 표시를 확인한다. 2. 냉각팬 회전을 확인한다. 3. 방열판 먼지를 제거한다.
#### 4.2.4 권장 조치
필터 압력차가 기준을 넘으면 흡기 필터를 교체한다. 냉각팬이 멈췄으면 냉각팬 모듈(PN-CF-3105)을 교체한다.
"""

practice_length_chunks = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=0).split_text(PRACTICE_MANUAL)
practice_mixed = 0
for practice_chunk in practice_length_chunks:
    practice_numbers = sorted(set(re.findall(r"\b\d\.\d\.\d\b", practice_chunk)))
    practice_mixed += len(practice_numbers) != 1
    print(f"  절 번호 {practice_numbers or '없음'} · {practice_chunk[:40]!r}")
print(f"조각 {len(practice_length_chunks)}개 중 인용할 절을 정할 수 없는 조각 {practice_mixed}개")

**결과 해설** — 120자로 자르면 조각 5개 중 2개는 인용할 절을 정할 수 없습니다. 셋째 조각에는 4.2.2 와 4.2.3 머리말이 함께 들어 있고, 마지막 조각은 머리말 없이 본문(권장 조치)만 남았습니다. 숫자에 안 잡힌 문제도 있습니다. 넷째 조각은 **4.2.3 의 점검 절차 본문**이 4.2.4 머리말과 붙어 있어, 번호만 보면 4.2.4 로 잘못 인용됩니다.

### 2단계 · 머리말로 자르고 인용 번호 만들기

In [ ]:
# MarkdownHeaderTextSplitter 는 머리말(#~####)을 경계로 자르고, 머리말을 메타데이터로 남깁니다.
# 가장 깊은 머리말의 번호가 곧 인용 번호입니다. 예: "4.2.3 점검 절차" → "MC01-MM 4.2.3"
# 관찰 포인트: 청크 앞에 머리말 경로를 붙입니다. 본문에 없는 "HDF" 가 경로에는 있습니다.
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter

practice_splitter = MarkdownHeaderTextSplitter([("#", "h1"), ("##", "h2"), ("###", "h3"), ("####", "h4")])

def practice_citation(meta):
    deepest = meta.get("h4") or meta.get("h3") or meta.get("h2") or ""
    number = re.match(r"(\d+(?:\.\d+)*)", deepest)
    return f"MC01-MM {number.group(1)}" if number else None

practice_sections = []
for practice_doc in practice_splitter.split_text(PRACTICE_MANUAL):
    practice_path = " > ".join(practice_doc.metadata[k] for k in ("h1", "h2", "h3", "h4") if k in practice_doc.metadata)
    practice_sections.append(Document(page_content=f"[{practice_path}]\n{practice_doc.page_content}",
                                      metadata={"citation": practice_citation(practice_doc.metadata),
                                                "body": practice_doc.page_content}))
for practice_doc in practice_sections:
    print(f"  {practice_doc.metadata['citation']:14} {practice_doc.page_content.splitlines()[0]}")

**결과 해설** — 본문이 있는 절 다섯 개가 각각 한 조각이 되고, 조각마다 인용 번호가 붙었습니다. 머리말만 있고 본문이 없는 4장·4.1·4.2 는 조각이 되지 않습니다. 이제 "이 근거는 어느 절인가"가 메타데이터 하나로 정해집니다.

### 3단계 · 머리말 경로가 검색을 살린다

In [ ]:
# 같은 조각을 본문만으로, 그리고 머리말 경로를 붙여서 BM25 로 찾아 봅니다.
# 토크나이저는 한글·영문·숫자 덩어리를 단어로 봅니다. "(HDF)" 에서 "HDF" 를 꺼냅니다.
# 관찰 포인트: 점검 절차 본문에는 "HDF" 도 "점검 절차" 도 없습니다. 그 말은 머리말에만 있습니다.
from langchain_community.retrievers import BM25Retriever

def practice_words(text):
    return re.findall(r"[0-9A-Za-z가-힣]+", text)

practice_query = "HDF 점검 절차"
practice_body_only = BM25Retriever.from_documents(
    [Document(page_content=d.metadata["body"], metadata=d.metadata) for d in practice_sections],
    preprocess_func=practice_words, k=1)
practice_with_path = BM25Retriever.from_documents(practice_sections, preprocess_func=practice_words, k=1)

practice_top_body = practice_body_only.invoke(practice_query)[0].metadata["citation"]
practice_top_path = practice_with_path.invoke(practice_query)[0].metadata["citation"]
print(f"본문만        → {practice_top_body}")
print(f"머리말 경로 붙임 → {practice_top_path}")

**결과 해설** — 본문만으로 찾으면 질의 단어 셋이 어느 본문에도 없어 순위에 근거가 없고, 엉뚱한 4.2.4(권장 조치)가 1위로 나옵니다. 머리말 경로를 붙이면 "HDF"·"점검"·"절차" 가 모두 맞는 4.2.3 이 1위가 됩니다. 실제 매뉴얼에서도 BM25 만으로 유형 절을 놓친 질의가 126개 중 40개에서 24개로 줄었습니다.

### 4단계 · PDF 로 읽으면

In [ ]:
# 현장에서 받는 매뉴얼은 대부분 PDF 입니다. 이 저장소의 PDF 변환본을 실제로 읽어 봅니다.
# Jupyter 는 Notebook 폴더에서, Test 는 저장소 루트에서 돕니다. 어느 쪽이든 저장소 루트를 찾습니다.
# 관찰 포인트: PDF 글자에는 머리말 표시(#)가 없습니다. 같은 자르기 규칙이 절을 하나도 찾지 못합니다.
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

practice_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "task10_maintenance").is_dir())
practice_pdf_pages = PyPDFLoader(str(practice_root / "task10_maintenance" / "data" / "manuals" / "pdf" / "MC01-MM.pdf")).load()
practice_pdf_text = "\n".join(page.page_content for page in practice_pdf_pages)
practice_pdf_docs = practice_splitter.split_text(practice_pdf_text)
practice_pdf_cited = [d for d in practice_pdf_docs if practice_citation(d.metadata)]
print("PDF 쪽 수:", len(practice_pdf_pages), "· 첫 쪽 메타데이터:", sorted(practice_pdf_pages[0].metadata)[:4])
print("'#' 로 시작하는 줄:", sum(line.startswith("#") for line in practice_pdf_text.splitlines()))
print("절 번호가 붙은 조각:", len(practice_pdf_cited), "/", len(practice_pdf_docs))

assert practice_mixed >= 1, "글자 수 자르기가 절을 섞지 않았습니다"
assert all(d.metadata["citation"] for d in practice_sections)
assert [d.metadata["citation"] for d in practice_sections] == ["MC01-MM 4.1.2", "MC01-MM 4.1.3", "MC01-MM 4.2.2", "MC01-MM 4.2.3", "MC01-MM 4.2.4"]
assert practice_top_path == "MC01-MM 4.2.3" and practice_top_body != "MC01-MM 4.2.3"
assert len(practice_pdf_pages) >= 1 and practice_pdf_cited == [], "PDF 에서 절 번호가 살아났습니다"

**결과 해설** — PDF 는 쪽마다 문서 하나가 되고 메타데이터에는 쪽 번호뿐입니다. 머리말 표시가 사라져 절 번호가 붙은 조각이 0개입니다. 그래서 이 자료는 **md 를 정본으로, PDF 를 변환본으로** 둡니다. PDF 로만 적재하면 인용은 쪽 단위(`MC01-MM p.8`)로 물러나고, 초안 검증이 절 번호를 확인할 수 없습니다.

## 직접 해 볼 과제
`practice_try_query` 를 "냉각팬 교체", "PN-TL-1008", "회전수 기준" 으로 바꿔 보세요. 머리말 경로를 붙인 쪽과 본문만 쓴 쪽이 각각 어느 절을 찾습니까? 어떤 질의에서 둘이 같아집니까?

In [ ]:
# 질의를 바꿔 두 검색기를 비교합니다. 조각과 토크나이저는 그대로입니다.
# 본문에 그 말이 있으면 둘이 같아지고, 머리말에만 있으면 갈립니다.
practice_try_query = "냉각팬 교체"
print("본문만        →", practice_body_only.invoke(practice_try_query)[0].metadata["citation"])
print("머리말 경로 붙임 →", practice_with_path.invoke(practice_try_query)[0].metadata["citation"])

## 중간 결과
글자 수 자르기에서 절이 섞인 조각 수, 절 단위 조각의 인용 번호, 머리말 경로 유무에 따른 BM25 1위, 그리고 PDF 에서 절 번호가 붙은 조각 수(0)를 확인합니다.

## 실패 경계
PDF 에서 머리말을 추정하는 규칙("숫자로 시작하는 줄")은 늘 틀립니다. 실제 MC01-MM PDF 에서 숫자로 시작하는 줄 105개 중 59개가 머리말이 아니라 번호 목록이었습니다. 원본이 PDF 뿐이라면 쪽 단위 인용으로 물러서거나, 구조를 복원하는 별도 단계가 필요합니다.

## 실제 app 연결
과제 10 의 적재 스크립트 `scripts/task10/ingest_manuals.py` 가 같은 방식(머리말 → 600자 재분할 → 머리말 경로 → 인용 번호)으로 매뉴얼 세 권을 69개 청크로 자릅니다. `--pdf` 로 돌리면 33개 청크가 나오고 절 번호가 붙은 것은 0개입니다. 앱은 `evidence.py::load_chunks` 로 md 청크를 읽고, `evidence.py::manual_section` 이 절 번호로 판정 기준 절을 바로 꺼냅니다.

## 다음 Notebook 연결
다음 `03_recorded_embeddings.ipynb` 에서는 이 조각을 vector 로 바꾸고, 키 없이도 같은 vector 를 쓰는 방법을 봅니다.